# Cella 1 - Installazione dipendenze

In [2]:
!apt-get install -q -y ffmpeg
!pip install -q rapidfuzz sentence-transformers
!pip install -q transformers>=4.40.0 accelerate librosa
!pip install json-repair

!pip install -q hydra-core lightning panphon phonemizer
!pip install -q torchaudio huggingface_hub

!pip install -q spacy
!python -m spacy download it_core_news_lg -q
!pip install pymongo

!pip install -q google-api-python-client

!pip install -q sacrebleu rouge-score jiwer

from sacrebleu.metrics import BLEU
from rouge_score import rouge_scorer
from jiwer import wer

from googleapiclient.discovery import build
from pymongo import MongoClient
from kaggle_secrets import UserSecretsClient
import uuid
from datetime import datetime, timezone

from pathlib import Path
import json as _json

import traceback


from transformers import BertTokenizerFast, BertForMaskedLM
import math



import os
import sys
import time
import string
import torch
import numpy as np
import pandas as pd
import librosa
import json
import gc
from transformers import (
    AutoModelForSpeechSeq2Seq,
    AutoProcessor,
    pipeline,
)

import copy

import re
from rapidfuzz import fuzz, process
from sentence_transformers import SentenceTransformer

import torchaudio
from huggingface_hub import hf_hub_download
from openai import OpenAI
import json_repair
import spacy

from openai import OpenAI

print('Dipendenze installate')

Reading package lists...
Building dependency tree...
Reading state information...
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 141 not upgraded.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 567.9/567.9 MB 2.8 MB/s eta 0:00:0000:0100:01
✔ Download and installation successful
You can now load the package via spacy.load('it_core_news_lg')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
Dipendenze installate


# Cella 2.1 - Verifica GPU

In [3]:

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
dtype = torch.float16 if DEVICE == "cuda" else torch.float32

if DEVICE == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('Nessuna GPU — Whisper large sara lento, considera whisper-base')

Device: cuda
GPU: Tesla T4
VRAM: 15.6 GB


# Cella 2.2 - Download Modelli

In [4]:
# Processore per model (large-v3 — 128 canali mel)
processor = AutoProcessor.from_pretrained("openai/whisper-large-v3")

model = AutoModelForSpeechSeq2Seq.from_pretrained(
    "openai/whisper-large-v3",
    torch_dtype=dtype,
    low_cpu_mem_usage=True,
    use_safetensors=True,
).to(DEVICE).eval()


!git clone https://github.com/changelinglab/PhoneticXeus.git
sys.path.append("/kaggle/working/PhoneticXeus")

from src.model.xeusphoneme.builders import build_xeus_pr_inference

REPO = "changelinglab/PhoneticXeus"
ckpt_path = hf_hub_download(REPO, "phoneticxeus_state_dict.pt")
# Il file vocab in realtà si trova già nella cartella scaricata da GitHub
vocab_path = "/kaggle/working/PhoneticXeus/src/model/xeusphoneme/resources/ipa_vocab.json"

    # Costruiamo l'inferenza rimuovendo ctc_weight e altri argomenti non supportati
inference = build_xeus_pr_inference(
work_dir="exp/cache/xeus",
hf_repo="espnet/xeus",
checkpoint=ckpt_path,
vocab_file=vocab_path,
device=DEVICE,
interctc_use_conditioning=True, # Questo è l'unico parametro extra supportato qui
)

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/1259 [00:00<?, ?it/s]

fatal: destination path 'PhoneticXeus' already exists and is not an empty directory.


Returning existing local_dir `exp/cache/xeus` as remote repo cannot be accessed in `snapshot_download` (None).


Loaded checkpoint: /root/.cache/huggingface/hub/models--changelinglab--PhoneticXeus/snapshots/bf28fd7958b9c20a268f7e93ce62ee1748713889/phoneticxeus_state_dict.pt with load info: <All keys matched successfully>


In [ ]:
# ── Lista di API KEY NVIDIA (aggiungine quante vuoi) ──────────
NVIDIA_API_KEYS = [
    "nvapi-key-1",
    "nvapi-key-2",
    "nvapi-key-3",
    "...",
]

_current_key_index = 0

def _get_nvidia_client() -> OpenAI:
    """Restituisce un client NVIDIA con la chiave attiva."""
    return OpenAI(
        base_url="https://integrate.api.nvidia.com/v1",
        api_key=NVIDIA_API_KEYS[_current_key_index],
    )

def _rotate_key():
    """Ruota alla chiave successiva. Solleva eccezione se le ha esaurite tutte."""
    global _current_key_index
    _current_key_index += 1
    if _current_key_index >= len(NVIDIA_API_KEYS):
        _current_key_index = 0  # oppure: raise RuntimeError("Tutte le API KEY esaurite")
        raise RuntimeError("Tutte le API KEY NVIDIA hanno ricevuto 429 — riprova più tardi.")
    print(f"  🔄 Rotazione chiave NVIDIA → indice {_current_key_index}")

MODEL = "mistralai/mistral-large-3-675b-instruct-2512"

# Cella 2.3 - Caricamento Modello BERT Italiano (PPPL)

In [6]:
# ══════════════════════════════════════════════════════════════
# PPPL — Pseudo-Perplexity su testo italiano normalizzato
# Modello: dbmdz/bert-base-italian-cased
# Motivazione: MLM monolingue italiano, cased per termini medici
# Formula: PPPL = exp( -1/N * sum_i log P(w_i | w_{\i}) )
# ══════════════════════════════════════════════════════════════

PPPL_MODEL_NAME = "dbmdz/bert-base-italian-cased"

print(f"Caricamento BERT italiano per PPPL: {PPPL_MODEL_NAME}")
_pppl_tokenizer = BertTokenizerFast.from_pretrained(PPPL_MODEL_NAME)
_pppl_model     = BertForMaskedLM.from_pretrained(PPPL_MODEL_NAME)
_pppl_model.eval()

# Sposta su GPU se disponibile — il modello è leggero (110M params)
_pppl_model = _pppl_model.to(DEVICE)
print(f"✅ Modello PPPL caricato su {DEVICE}")

def _normalize_pppl(pppl_value: float,
                    n_tokens: int = None,
                    pppl_min: float = 5.0,
                    pppl_max: float = 500.0) -> float:
    if pppl_value is None:
        return 0.5
    # Testo troppo corto: PPPL non ha significato statistico, valore neutro
    if n_tokens is not None and n_tokens < 3:
        return 0.1
    normed = (pppl_value - pppl_min) / (pppl_max - pppl_min)
    return float(np.clip(normed, 0.0, 1.0))

def compute_pppl(text: str,
                 tokenizer=_pppl_tokenizer,
                 model=_pppl_model,
                 pppl_min: float = 5.0,
                 pppl_max: float = 500.0,
                 max_length: int = 512) -> dict:
    """
    Calcola la Pseudo-Perplexity (PPPL) su un testo italiano normalizzato.

    Algoritmo:
      Per ogni token i nella sequenza:
        1. Maschera il token i con [MASK]
        2. Ottieni P(w_i | w_{\\i}) dalla testa MLM di BERT
        3. Accumula log P
      PPPL = exp( -1/N * sum log P )

    Interpretazione:
      - PPPL bassa  → testo fluente, italiano standard atteso
      - PPPL alta   → testo insolito, anomalie semantiche o errori ASR residui

    Args:
      text       : testo italiano normalizzato (output di analyze_and_normalize_with_llm)
      tokenizer  : BertTokenizerFast precaricato
      model      : BertForMaskedLM precaricato
      max_length : lunghezza massima token (default 512, limite BERT)

    Returns:
      dict con:
        'pppl'         : float  — valore PPPL (più basso = migliore)
        'log_pppl'     : float  — log2(PPPL) per confronti su scale diverse
        'n_tokens'     : int    — numero di token analizzati
        'mean_log_prob': float  — media dei log P (prima di exp, negato)
        'model'        : str    — nome modello usato
    """
    if not text or not text.strip():
        return {
            'pppl':          None,
            'log_pppl':      None,
            'n_tokens':      0,
            'mean_log_prob': None,
            'model':         PPPL_MODEL_NAME,
        }

    # Tokenizza senza truncation prima per verificare lunghezza
    encodings = tokenizer(
        text,
        return_tensors='pt',
        truncation=True,
        max_length=max_length,
    )

    input_ids = encodings['input_ids'].to(DEVICE)  # shape: (1, seq_len)
    seq_len   = input_ids.shape[1]

    # I token speciali [CLS] e [SEP] non vengono mascherati
    # (indici 0 e seq_len-1)
    mask_token_id = tokenizer.mask_token_id
    total_log_prob = 0.0
    n_tokens       = 0

    with torch.no_grad():
        for i in range(1, seq_len - 1):  # esclude [CLS] e [SEP]
            # Crea una copia e maschera la posizione i
            masked_ids       = input_ids.clone()
            masked_ids[0, i] = mask_token_id

            outputs = model(masked_ids)
            logits  = outputs.logits  # shape: (1, seq_len, vocab_size)

            # Log-softmax sulla distribuzione al token mascherato
            log_probs    = torch.nn.functional.log_softmax(logits[0, i], dim=-1)
            true_token   = input_ids[0, i].item()
            log_prob_tok = log_probs[true_token].item()

            total_log_prob += log_prob_tok
            n_tokens       += 1

    if n_tokens == 0:
        return {
            'pppl':          None,
            'log_pppl':      None,
            'n_tokens':      0,
            'mean_log_prob': None,
            'model':         PPPL_MODEL_NAME,
        }

    mean_log_prob = total_log_prob / n_tokens          # negativo
    pppl          = round(math.exp(-mean_log_prob), 4) # exp(-(-|x|)) = exp(|x|)
    log_pppl      = round(math.log2(pppl), 4) if pppl > 0 else None

    pppl_norm = _normalize_pppl(pppl, n_tokens=n_tokens, pppl_min=pppl_min, pppl_max=pppl_max)

    return {
        'pppl':          pppl,
        'log_pppl':      log_pppl,
        'n_tokens':      n_tokens,
        'mean_log_prob': round(mean_log_prob, 6),
        'pppl_norm':     pppl_norm,
        'model':         PPPL_MODEL_NAME,
    }


print("✅ Funzione compute_pppl definita")


Caricamento BERT italiano per PPPL: dbmdz/bert-base-italian-cased


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

BertForMaskedLM LOAD REPORT from: dbmdz/bert-base-italian-cased
Key                         | Status     |  | 
----------------------------+------------+--+-
bert.pooler.dense.weight    | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 
bert.pooler.dense.bias      | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Modello PPPL caricato su cuda
✅ Funzione compute_pppl definita


# Cella 3.1 - Configurazione

In [7]:
CONFIG = {
    # Embedding multilingue leggero — gira su CPU, non occupa VRAM
    'embedding_model': 'paraphrase-multilingual-MiniLM-L12-v2',

    # Quante frasi napoletane nel prompt dinamico
    'num_frasi_prompt': 4,

    # Soglia fuzzy matching (0-100)
    'fuzzy_threshold': 72,

    # Limite caratteri prompt (Whisper: ~224 token ~ 800 chars)
    'max_prompt_chars': 800,
}

print('Config caricata')

Config caricata


# Cella 3.2 - Configurazione MongoDB Atlas

In [8]:
user_secrets = UserSecretsClient()
MONGO_URI = user_secrets.get_secret("MONGO_URI_GIGGIONE")

client = MongoClient(MONGO_URI)
db = client['asr_dialects_db_no_rags']

col_sessions            = db['Sessions']
col_transcripts         = db['Transcripts']
col_risk_logs           = db['Risk_Logs']
col_pipeline_stages     = db['Pipeline_Stages']
col_translation_metrics = db['Translation_Metrics']
col_knowledge           = db['Dialect_KnowledgeBase']
recordings_col          = db['recordings']


print("✅ Connesso a MongoDB Atlas e collezioni inizializzate!")
print(f"   Documenti in Dialect_KnowledgeBase: {col_knowledge.count_documents({})}")

✅ Connesso a MongoDB Atlas e collezioni inizializzate!
   Documenti in Dialect_KnowledgeBase: 0


# Cella 3.3 - Popolamento MongoDB Atlas (Dialect_KnowledgeBase)

# Cella 3.4 - Gestione Response LLM e salvataggi MongoDB

In [9]:
MAX_LLM_RETRIES = 2
LLM_RETRY_DELAY = 2

def call_llm_with_retry(
    prompt: str,
    max_tokens: int = 2048,
    temperature: float = 0.1,
    system_prompt: str = None,
) -> str:
    messages = []
    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})
    messages.append({"role": "user", "content": prompt})

    last_error = None
    for attempt in range(1, MAX_LLM_RETRIES + 1):
        try:
            client = _get_nvidia_client()  # usa sempre la chiave attiva
            response = client.chat.completions.create(
                model=MODEL,
                messages=messages,
                temperature=temperature,
                max_tokens=max_tokens,
            )
            raw = response.choices[0].message.content.strip()
            if not raw:
                raise ValueError("Risposta LLM vuota")
            return raw

        except Exception as e:
            last_error = e
            err_str = str(e)
            print(f"  ⚠️  LLM tentativo {attempt}/{MAX_LLM_RETRIES} fallito: {e}")

            # ── Rotazione su 429 ─────────────────────────────────
            if "429" in err_str or "Too Many Requests" in err_str.lower():
                try:
                    _rotate_key()
                    continue  # ritenta subito con la nuova chiave
                except RuntimeError as re:
                    raise RuntimeError(str(re)) from e

            if attempt < MAX_LLM_RETRIES:
                time.sleep(LLM_RETRY_DELAY)

    raise RuntimeError(
        f"LLM non disponibile dopo {MAX_LLM_RETRIES} tentativi. "
        f"Ultimo errore: {last_error}"
    )

In [10]:
def _now_ms() -> int:
    """Unix timestamp in millisecondi (UTC). Unica funzione di tempo del progetto."""
    return int(datetime.now(timezone.utc).timestamp() * 1000)


# ── SESSIONS ────────────────────────────────────────────────────────────
def init_new_session(recording_doc: dict) -> str:
    recording_id   = recording_doc["_id"]
    participant_id = recording_doc["participantId"]

    session_doc = {
        "_id":            recording_id,
        "participant_id": participant_id,
        "filename":       recording_doc.get("filename"),
        "promptId":       recording_doc.get("promptId"),
        "promptCategory": recording_doc.get("promptCategory"),
        "promptText":     recording_doc.get("promptText"),
        "started_at":     _now_ms(),
        "status":         "processing",
    }

    col_sessions.replace_one({"_id": recording_id}, session_doc, upsert=True)
    return recording_id


# ── TRANSCRIPTS ─────────────────────────────────────────────────────────
def save_transcript_output(session_id: str,
                           transcript_results: dict,
                           analysis_results: dict,
                           t_start: int,
                           t_end: int):
    col_transcripts.replace_one(
        {"_id": session_id},
        {
            "_id":       session_id,
            "started_at":  t_start,
            "saved_at":    t_end,
            "transcription_stage": {
            "mode":               transcript_results.get("mode", "whisper_standard_baseline"),
            "testo_finale":       transcript_results.get("testo_finale"),
            "model":              transcript_results.get("model"),
            },
            "analysis_stage": {
                "period_conf_mean": float(analysis_results.get("period_conf_mean", 0)),
                "period_conf_geo":  float(analysis_results.get("period_conf_geo", 0)),
                "threshold_used":   float(analysis_results.get("threshold_used", 0.70)),
                "tokens":           analysis_results.get("tokens"),
                "words":            analysis_results.get("words"),
                "low_conf_tokens":  analysis_results.get("low_conf_tokens"),
                "low_conf_words":   analysis_results.get("low_conf_words"),
                # ── PPPL (Pseudo-Perplexity) sul testo italiano normalizzato ──
            },
        },
        upsert=True
    )


# ── PIPELINE STAGES ─────────────────────────────────────────────────────
def save_pipeline_stage(session_id: str,
                        stage_name: str,
                        stage_data,
                        t_start: int,
                        t_end: int):
    col_pipeline_stages.update_one(
        {"_id": session_id},
        {
            "$set": {
                f"stages.{stage_name}": {
                    "data":       copy.deepcopy(stage_data),
                    "started_at": t_start,
                    "saved_at":   t_end,
                },
                "last_updated": t_end,
            }
        },
        upsert=True
    )


# ── RISK LOGS ────────────────────────────────────────────────────────────
def save_risk_scoring(session_id: str,
                      risk_scores_output: list,
                      t_start: int,
                      t_end: int):
    col_risk_logs.replace_one(
        {"_id": session_id},
        {
            "_id":          session_id,
            "started_at":   t_start,
            "saved_at":     t_end,
            "overall":      compute_overall_risk(risk_scores_output),
            "scored_claims": copy.deepcopy(risk_scores_output),
        },
        upsert=True
    )


# ── TRANSLATION METRICS ──────────────────────────────────────────────────
def save_translation_metrics(session_id: str,
                              metrics: dict,
                              t_start: int,
                              t_end: int):
    col_translation_metrics.replace_one(
        {"_id": session_id},
        {
            "_id":        session_id,
            "started_at": t_start,
            "saved_at":   t_end,
            **metrics,
        },
        upsert=True
    )


# ── COMPLETE SESSION ─────────────────────────────────────────────────────
def complete_session(session_id: str,
                     success: bool = True,
                     error_msg: str = None,
                     overall: dict = None):
    update = {
        "completed_at": _now_ms(),
        "status":       "completed" if success else "failed",
    }
    if error_msg:
        update["error_log"] = error_msg
    if overall is not None:
        update["overall_risk"] = overall

    col_sessions.update_one({"_id": session_id}, {"$set": update})

# Cella 4 - Caricamento dizionario napoletano da MongoDB Atlas

# Cella 5 - Retriever semantico (MongoDB $vectorSearch) + fuzzy

# Cella 6 - Whisper RAG + Calcolo confidenza per token

In [11]:
class WhisperStandardNapoletano:
    """
    Pipeline Whisper singola passata, senza RAG.
    Serve come baseline per validare l'effetto di Whisper-RAG.
    """

    def __init__(self, processor, model):
        self.processor = processor
        self.model = model

    def trascrivi(self, audio_array: np.ndarray, verbose: bool = False) -> dict:
        inputs = self.processor(
            audio_array,
            sampling_rate=16000,
            return_tensors="pt"
        )

        input_features = inputs.input_features.to(DEVICE, dtype=dtype)

        with torch.no_grad():
            generated = self.model.generate(
                input_features,
                return_dict_in_generate=True,
                output_scores=True,
                max_new_tokens=128,
                language="it",
                task="transcribe"
            )

        sequences = generated.sequences
        testo = self.processor.batch_decode(
            sequences,
            skip_special_tokens=True
        )[0].strip()

        transition_scores = self.model.compute_transition_scores(
            sequences,
            generated.scores,
            normalize_logits=True
        )

        token_ids = sequences[0]
        tokens = []
        confidences = []

        for token_id, logprob in zip(token_ids[-transition_scores.shape[1]:], transition_scores[0]):
            token_text = self.processor.tokenizer.decode([token_id])
            prob = float(torch.exp(logprob).detach().cpu())

            tokens.append({
                "token_id": int(token_id.detach().cpu()),
                "token_text": token_text,
                "confidence": prob,
                "logprob": float(logprob.detach().cpu()),
            })
            confidences.append(prob)

        period_conf_mean = float(np.mean(confidences)) if confidences else None
        period_conf_geo = float(np.exp(np.mean(np.log(np.clip(confidences, 1e-9, 1.0))))) if confidences else None

        return {
            "testo_finale": testo,
            "result": {
                "text": testo,
                "tokens": tokens,
                "period_conf_mean": period_conf_mean,
                "period_conf_geo": period_conf_geo,
            }
        }

# Cella 7 - Inizializzazione WHISPER

In [12]:
pipeline_whisper = WhisperStandardNapoletano(
    processor=processor,
    model=model,
)

# Embedder usato per metriche semantiche
translation_embedder = SentenceTransformer(
    CONFIG["embedding_model"],
    device="cpu"
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


# Cella 8 - Carica audio da Google Drive

In [13]:
user_secrets = UserSecretsClient()
API_KEY = user_secrets.get_secret("GDRIVE_API_KEY")
FOLDER_ID = "1jgHjXqFfGl_WVwAwkZwZ_hJaetTnw4xr"  # dall'URL della cartella condivisa

drive_service = build("drive", "v3", developerKey=API_KEY)


def get_all_files_in_folder(folder_id: str) -> dict:
    """
    Restituisce un dizionario {filename: drive_file_id}
    per tutti i WAV nella cartella e sottocartelle.
    """
    file_map = {}

    def _recurse(fid):
        page_token = None
        while True:
            response = drive_service.files().list(
                q=f"'{fid}' in parents and trashed=false",
                fields="nextPageToken, files(id, name, mimeType)",
                pageToken=page_token,
                supportsAllDrives=True,
                includeItemsFromAllDrives=True,
            ).execute()

            for item in response.get("files", []):
                if item["mimeType"] == "application/vnd.google-apps.folder":
                    _recurse(item["id"])  # scendi nelle sottocartelle
                elif item["name"].endswith(".wav"):
                    file_map[item["name"]] = item["id"]

            page_token = response.get("nextPageToken")
            if not page_token:
                break

    _recurse(folder_id)
    return file_map

print("Recupero file IDs da Google Drive...")
file_map = get_all_files_in_folder(FOLDER_ID)
print(f"Trovati {len(file_map)} file WAV")

# Aggiorna MongoDB con i drive_file_id
updated = 0
not_found = []

for doc in recordings_col.find({}):
    filename = doc.get("filename")
    if filename in file_map:
        recordings_col.update_one(
            {"_id": doc["_id"]},
            {"$set": {"drive_file_id": file_map[filename]}}
        )
        updated += 1
    else:
        not_found.append(filename)

print(f"✅ Aggiornati {updated} documenti con drive_file_id")
if not_found:
    print(f"⚠️  Non trovati su Drive: {not_found}")

Recupero file IDs da Google Drive...
Trovati 50 file WAV
✅ Aggiornati 50 documenti con drive_file_id


In [14]:
def download_from_drive_public(drive_file_id: str, local_path: str):
    """
    Scarica un file pubblico da Google Drive usando l'API Key.
    Gestisce il cookie di conferma per file grandi.
    """
    import requests

    session = requests.Session()
    URL = "https://docs.google.com/uc"
    params = {"export": "download", "id": drive_file_id}

    response = session.get(URL, params=params, stream=True)

    # Gestisci il token di conferma antivirus per file grandi
    token = None
    for key, value in response.cookies.items():
        if key.startswith("download_warning"):
            token = value
            break

    if token:
        params["confirm"] = token
        response = session.get(URL, params=params, stream=True)

    # Verifica che non sia una pagina HTML di errore
    content_type = response.headers.get("Content-Type", "")
    if "text/html" in content_type:
        raise ValueError(
            f"Drive ha restituito HTML. "
            f"Verifica che il file {drive_file_id} sia pubblico."
        )

    with open(local_path, "wb") as f:
        for chunk in response.iter_content(chunk_size=32768):
            if chunk:
                f.write(chunk)

    size_kb = os.path.getsize(local_path) / 1024
    print(f"✅ Scaricato: {Path(local_path).name} ({size_kb:.1f} KB)")

In [26]:
from pathlib import Path
TEMP_AUDIO_DIR = Path('/kaggle/working/temp_audios')
TEMP_AUDIO_DIR.mkdir(parents=True, exist_ok=True)


# ── Pulizia sessioni failed/processing ───────────────────────
statuses_da_ripulire = ["failed", "processing"]

sessioni_da_ripulire = list(col_sessions.find(
    {"status": {"$in": statuses_da_ripulire}},
    {"_id": 1}                          # ← recording_id non esiste più, solo _id
))

sessioni_ids_da_cancellare = [s["_id"] for s in sessioni_da_ripulire]

if sessioni_ids_da_cancellare:
    col_sessions.delete_many({"_id":            {"$in": sessioni_ids_da_cancellare}})
    col_transcripts.delete_many({"_id":         {"$in": sessioni_ids_da_cancellare}})
    col_risk_logs.delete_many({"_id":           {"$in": sessioni_ids_da_cancellare}})
    col_pipeline_stages.delete_many({"_id":     {"$in": sessioni_ids_da_cancellare}})
    col_translation_metrics.delete_many({"_id": {"$in": sessioni_ids_da_cancellare}})
    # ❌ col_history rimossa
    print(f"🗑️  Cancellate {len(sessioni_ids_da_cancellare)} sessioni ({statuses_da_ripulire})")
    print(f"🗑️  Cancellate tracce collegate in tutte le collezioni")
else:
    print("✅ Nessuna sessione failed/processing trovata")

# ── Query recordings: completati da saltare + failed da includere ──
sessioni_completate = set(
    doc["_id"] for doc in col_sessions.find({"status": "completed"}, {"_id": 1})
)

query  = {"_id": {"$nin": list(sessioni_completate)}}
cursor = recordings_col.find(query)

#Se vuoi diminuire le richieste usa skip e limit

print(f"📋 Recording da processare: {recordings_col.count_documents(query)}")
print(f"   (di cui {len(sessioni_ids_da_cancellare)} erano failed/processing e sono stati ripuliti)")

#query = {}
#cursor = recordings_col.find(query).skip(1).limit(6)

audio_files = []
audio_docs  = []

for doc in cursor:
    filename      = doc.get("filename")
    drive_file_id = doc.get("drive_file_id")

    if not drive_file_id:
        print(f"⚠️  Nessun drive_file_id per {filename}")
        continue

    local_path = str(TEMP_AUDIO_DIR / filename)

    # Salta il download se il file esiste già
    if os.path.exists(local_path):
        print(f"⏭️  Già presente: {filename}")
        audio_files.append(local_path)
        audio_docs.append(doc)
        continue

    try:
        print(f"📥 Download: {filename} ...")
        download_from_drive_public(drive_file_id, local_path)
        audio_files.append(local_path)
        audio_docs.append(doc)
    except Exception as e:
        print(f"❌ Errore download {filename}: {e}")

print(f"\nPronti: {len(audio_files)} file")

🗑️  Cancellate 1 sessioni (['failed', 'processing'])
🗑️  Cancellate tracce collegate in tutte le collezioni
📋 Recording da processare: 1
   (di cui 1 erano failed/processing e sono stati ripuliti)
⏭️  Già presente: prompt-5565_rec-10.wav

Pronti: 1 file


In [16]:
REPORTS_DIR = Path('/kaggle/working/reports')
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

def _json_default(o):
    if isinstance(o, (np.integer,)):  return int(o)
    if isinstance(o, (np.floating,)): return float(o)
    if isinstance(o, (np.ndarray,)):  return o.tolist()
    if isinstance(o, set):            return list(o)
    if isinstance(o, Path):           return str(o)
    try:    return str(o)
    except: return None

# Cella 9 - Calcolo Confidenza a livello di parole

In [17]:
def tokens_to_words(token_data: list) -> list:

    """
    Aggrega i token in parole e calcola la confidence per parola.
    Isola i segni di punteggiatura trattandoli come parole a sé stanti.
    """

    words = []
    current_tokens = []
    # Set di caratteri di punteggiatura da isolare
    PUNCTUATION = set(string.punctuation) # Include !"#$%&'()*+,-./:;<=>?@[\]^_`{|}~

    for token in token_data:
        text = token["token_text"]
        clean_text = text.strip()
        # 1. È il primo token in assoluto?
        is_first_token = len(current_tokens) == 0

        # 2. Inizia con uno spazio? (Classico inizio parola in Whisper)
        starts_with_space = text.startswith(" ")

        # 3. Il token corrente è composto SOLO da punteggiatura? (es. "?", "...", "!")
        is_punctuation = len(clean_text) > 0 and all(char in PUNCTUATION for char in clean_text)

        # 4. L'ultimo token inserito era punteggiatura?
        # (Serve per staccare la parola successiva anche se manca lo spazio)
        prev_is_punctuation = False
        if not is_first_token:
            prev_text = current_tokens[-1]["token_text"].strip()
            prev_is_punctuation = len(prev_text) > 0 and all(char in PUNCTUATION for char in prev_text)
        # Scatta la separazione se si verifica una qualsiasi di queste condizioni
        is_new_word = is_first_token or starts_with_space or is_punctuation or prev_is_punctuation
        if is_new_word and current_tokens:
            # Chiudi la parola/punteggiatura corrente prima di aprirne una nuova
            words.append(_aggregate_word(current_tokens))
            current_tokens = []

        current_tokens.append(token)

    # Ultima parola rimasta
    if current_tokens:
        words.append(_aggregate_word(current_tokens))

    return words


def _aggregate_word(tokens: list) -> dict:
    """Calcola le statistiche di confidence per un gruppo di token."""
    word_text   = "".join(t["token_text"] for t in tokens).strip()
    confidences = [t["confidence"] for t in tokens]
    clipped     = np.clip(confidences, 1e-9, 1.0)

    return {
        "word":         word_text,
        "n_tokens":     len(tokens),
        "tokens":       [t["token_text"] for t in tokens],
        "conf_mean":    round(float(np.mean(confidences)),          6),
        "conf_min":     round(float(np.min(confidences)),           6),
        "conf_geo":     round(float(np.exp(np.mean(np.log(clipped)))), 6),
        "conf_product": round(float(np.prod(clipped)),              6),
    }

# Cella 10 - Trascrizione Fonetica con PhoneticXeus

In [18]:
def transcribe_with_phonetic_xeus_fixed(audio_path, inference):

    print(f"3. Caricamento e preprocessing dell'audio: {audio_path}")
    # Usiamo torchaudio come richiesto dal loro sistema
    waveform, sr = torchaudio.load(audio_path)

    # Forziamo il campionamento a 16kHz se l'audio originale è diverso
    if sr != 16000:
        waveform = torchaudio.functional.resample(waveform, sr, 16000)

    print("4. Inferenza in corso...")
    # Il modello si aspetta un tensore 1D, quindi usiamo squeeze(0) per rimuovere la dimensione dei canali
    results = inference(waveform.squeeze(0))

    # L'output è una lista di dizionari. La trascrizione pulita è sotto la chiave "processed_transcript"
    transcription = results[0]["processed_transcript"]

    return transcription

# Cella 12 - Analisi Trascrizioni LLM (traduzione napoletano + normralizzazione italiano + parole risolte/problematiche)

In [19]:
def analyze_and_normalize_with_llm(
    transcript_whisper: str,
    transcript_PhoneticXeus: str,
    word_data: list,
) -> dict:
    """
    Versione SENZA analisi sintattica LLM e SENZA RAG.

    Il LLM non produce più:
      - is_dialectal
      - dialectal_type
      - surprisal

    Esegue solo:
      - domain detection
      - semantic issue detection
      - normalizzazione in italiano

    Le parole Whisper vengono passate solo come contesto con confidence.
    """

    words_meta = []
    for i, w in enumerate(word_data):
        words_meta.append({
            "index": i,
            "word": w["word"],
            "conf_mean": round(w.get("conf_mean", 0.0), 3),
            "conf_min": round(w.get("conf_min", 0.0), 3),
        })

    words_meta_json = json.dumps(words_meta, ensure_ascii=False)

    system_prompt = """
You are an expert linguist specialized in Southern Italian dialects
(Campanian / Neapolitan) and an expert in clinical and pharmacological language.

Your task is to reconstruct and normalize noisy speech transcriptions into fluent,
natural Italian by triangulating evidence across multiple sources.

══════════════════════════════════════════════════════════════
CORE OBJECTIVE
══════════════════════════════════════════════════════════════

Your goal is NOT to aggressively correct the transcript.

Your goal is to identify ONLY genuinely suspicious tokens and resolve them
through evidence triangulation.

Never hallucinate words.
Never invent drugs, symptoms, entities or dialectal forms.
Never force corrections without strong evidence.

If evidence is insufficient:
PRESERVE the original Whisper token.

══════════════════════════════════════════════════════════════
AVAILABLE SOURCES
══════════════════════════════════════════════════════════════

1. WHISPER TRANSCRIPTION

Main ASR transcription.

This is the primary textual source.

2. WHISPER WORD METADATA

The full ordered Whisper word list is provided.

Each word includes:
- word index
- surface word
- conf_mean
- conf_min

Low confidence values should receive higher attention,
but confidence alone is never sufficient to modify a token.

Confidence values indicate ASR reliability,
not semantic correctness.

3. PHONETICXEUS (IPA)

Language-independent phonetic transcription of the actual spoken sounds.

IMPORTANT:
- Do NOT perform literal IPA string matching.
- IPA is phonetic evidence only.
- Focus on:
  - rhythm
  - consonant structure
  - stressed vowels
  - sound presence/absence
- PhoneticXeus may merge adjacent words into continuous strings.
- IPA alignment is approximate, not positional.

══════════════════════════════════════════════════════════════
TRIANGULATION PRIORITY
══════════════════════════════════════════════════════════════

When evidence conflicts, use this priority order:

1. Strong IPA evidence
2. Semantic coherence
3. Whisper acoustic plausibility
4. Word confidence metadata

Confidence values are supporting evidence only.

A low-confidence token is not necessarily wrong.
A high-confidence token is not necessarily correct.

══════════════════════════════════════════════════════════════
PHASE 1 — DOMAIN DETECTION
══════════════════════════════════════════════════════════════

Classify the transcription into ONE domain:

- medico
- farmacologico
- quotidiano
- emotivo
- altro

IMPORTANT:

Medical rules apply ONLY if the domain is:
- medico
- farmacologico

Do NOT force medical interpretations in non-medical contexts.

Example:

"pressione" in an emotional context may refer to:
- stress
- anxiety

and not necessarily blood pressure.

══════════════════════════════════════════════════════════════
PHASE 2 — SEMANTIC TRIANGULATION
══════════════════════════════════════════════════════════════

──────────────────────────────────────────────────────────────
LOCAL CONTEXT AND SEMANTIC INTERPRETATION
──────────────────────────────────────────────────────────────

When analyzing a token for semantic issues, do NOT evaluate it in isolation.

Always consider:

- previous words
- following words
- whether adjacent words form a semantic unit
- whether a token is part of an article+noun fusion
- whether consecutive tokens are fragments of a single word
- whether meaning becomes clear only through local context
- whether surrounding words support a medical,
  pharmacological, emotional or everyday interpretation

The semantic_issues analysis must be phrase-aware,
not token-isolated.

Do NOT translate word by word.

Normalization must preserve the intended meaning of the utterance,
not the literal surface form of each token.

If a dialectal expression or noisy transcription corresponds to a clear
Italian meaning, normalize it semantically.

Examples:

- "mal e capo" may correspond to "mal di testa" if context supports it.
- "tachi pirina" may correspond to "Tachipirina" only if IPA and context support it.
- "pressione" should not automatically mean blood pressure unless context supports it.

──────────────────────────────────────────────────────────────
STEP 2.1 — ERROR DETECTION & TRIANGULATION
──────────────────────────────────────────────────────────────

Identify ONLY genuinely suspicious or conflicting tokens.

A token is suspicious ONLY if at least one applies:

- low conf_min
- low conf_mean
- semantic inconsistency
- article+word fusion
- token fragmentation
- strong IPA mismatch
- local phrase does not make sense

Do NOT assume that a token is incorrect solely because of low confidence.

Always evaluate:

- local sentence context
- IPA evidence
- semantic coherence
- confidence metadata

Do NOT modify tokens simply because they appear dialectal.

For every problematic token:

1. Explain why it is suspicious
2. Mention relevant confidence values when useful
3. Evaluate IPA evidence
4. Determine the most plausible correction
5. Explain why the correction is semantically and phonetically justified

The "reason" field MUST explicitly mention, when relevant:

- conf_min
- conf_mean
- IPA evidence
- semantic coherence
- domain coherence

══════════════════════════════════════════════════════════════
SPECIAL RECONSTRUCTION RULES
══════════════════════════════════════════════════════════════

──────────────────────────────────────────────────────────────
MULTI-SOURCE CONVERGENCE
──────────────────────────────────────────────────────────────

If:

- Whisper transcription
- confidence metadata
- IPA evidence

are mutually coherent,

KEEP the token unchanged,
even if unusual or dialectal.

──────────────────────────────────────────────────────────────
ARTICLE + WORD FUSION
──────────────────────────────────────────────────────────────

Whisper may merge:

- l'+word
- dell'+word
- nell'+word
- sull'+word

Examples:

- losso → l'osso
- laringuine → l'inguine

Apply ONLY if:

- IPA supports vowel onset
- resulting split forms a real coherent word
- local context supports the reconstruction

──────────────────────────────────────────────────────────────
FRAGMENTATION / DETOKENIZATION
──────────────────────────────────────────────────────────────

Whisper may split long words.

Examples:

- tachi pirina
- mal e capo

If IPA suggests a continuous phonetic sequence,
reconstruct the unified token.

──────────────────────────────────────────────────────────────
RESIDUAL UNCERTAINTY
──────────────────────────────────────────────────────────────

If ambiguity remains unresolved:

- mark the issue as uncertain
- preserve the original Whisper token
- NEVER invent words

══════════════════════════════════════════════════════════════
MEDICAL REALITY CHECK
══════════════════════════════════════════════════════════════

Apply ONLY if the detected domain is:

- medico
- farmacologico

Rules:

1. Never invent nonexistent drugs.

2. Normalize distorted drug names ONLY if strongly supported.

3. Do NOT force literal IPA matching.

4. IPA alignment is phonetic, not character-based.

5. Proper names are NOT medical entities,
even if phonetically similar.

Examples:

- pirini → Aspirina
- tachi pirina → Tachipirina
- brufe → Brufen

ONLY when context and IPA strongly support the correction.

══════════════════════════════════════════════════════════════
STEP 2.2 — NORMALIZATION
══════════════════════════════════════════════════════════════

Produce fluent standard Italian.

IMPORTANT:

Correct ONLY tokens identified as problematic in STEP 2.1.

If a token was NOT flagged as problematic,
it is considered correct by definition.

Rules:

- preserve non-problematic tokens
- normalize dialect naturally
- normalize idioms semantically
- avoid over-correction
- preserve original meaning
- preserve original tone

The final sentence must sound natural to a native Italian speaker.

══════════════════════════════════════════════════════════════
FEW-SHOT EXAMPLES
══════════════════════════════════════════════════════════════

──────────────────────────────────────────────────────────────
EXAMPLE 1 — Article fusion
──────────────────────────────────────────────────────────────

Whisper:
"Me fa male losso d'a gamma"

IPA:
"mefamallelɔssodaggamma"

Analysis:

- "losso" is suspicious
- IPA supports a continuous sequence compatible with "l'osso"
- local context is anatomically coherent

Normalized:

"Mi fa male l'osso della gamba."

──────────────────────────────────────────────────────────────
EXAMPLE 2 — Fragmentation / Detokenization
──────────────────────────────────────────────────────────────

Whisper:
"Aggi' pigliato 'a tachi pirina"

IPA:
"addʒipiʎʎatotakipirina"

Analysis:

- fragmented consecutive tokens
- IPA suggests a continuous sequence
- pharmacological entity recognized

Normalized:

"Ho preso la Tachipirina."

──────────────────────────────────────────────────────────────
EXAMPLE 3 — Semantic normalization
──────────────────────────────────────────────────────────────

Whisper:
"Aggio nu mal e capo forte"

IPA:
"addʒonumalekapoforte"

Analysis:

- local phrase suggests a known symptom
- IPA supports continuity
- semantic interpretation is unambiguous

Normalized:

"Ho un forte mal di testa."

══════════════════════════════════════════════════════════════
OUTPUT FORMAT
══════════════════════════════════════════════════════════════

For every semantic issue, return the original Whisper word index/indices.

Indices must refer to the position of the token in the full Whisper word list,
using 0-based indexing.

If the issue involves multiple adjacent tokens,
return all involved indices.

Return ONLY valid JSON.

STRICT RULES

- no markdown
- no explanations outside JSON
- no trailing commas
- use valid JSON syntax only
- use double quotes for JSON syntax

Required schema:

{
  "normalization": {
    "detected_domain": "medico | farmacologico | quotidiano | emotivo | altro",
    "semantic_issues": [
      {
        "words": ["token"],
        "word_indices": [12],
        "reason": "Explain the issue using confidence metadata, IPA evidence and semantic/domain coherence."
      }
    ],
    "normalized_text": "Italian normalized sentence"
  }
}
"""

    user_prompt = f"""
Analyze the following transcription set.

WHISPER TRANSCRIPTION:
"{transcript_whisper}"

PHONETICXEUS IPA:
"{transcript_PhoneticXeus}"

ORDERED WHISPER WORD METADATA:
{words_meta_json}

Tasks:
1. Detect the semantic domain
2. Identify problematic tokens through semantic and phonetic triangulation
3. Normalize the transcription into fluent standard Italian

Return ONLY the final JSON object.
"""

    raw = None
    result = None

    for attempt in range(1, MAX_LLM_RETRIES + 1):
        try:
            raw = call_llm_with_retry(
                prompt=user_prompt,
                system_prompt=system_prompt,
                max_tokens=4096,
                temperature=0.1,
            )

            clean = raw
            if clean.startswith("```json"):
                clean = clean[7:]
            if clean.startswith("```"):
                clean = clean[3:]
            if clean.endswith("```"):
                clean = clean[:-3]
            clean = clean.strip()

            try:
                result = json.loads(clean)
            except json.JSONDecodeError:
                result = json_repair.loads(clean)

            if "normalization" not in result:
                raise ValueError("Campo 'normalization' mancante")
            if "normalized_text" not in result["normalization"]:
                raise ValueError("Campo 'normalized_text' mancante in normalization")

            break

        except Exception as e:
            print(f"  ⚠️ analyze_and_normalize tentativo {attempt}/{MAX_LLM_RETRIES} fallito: {e}")
            if attempt >= MAX_LLM_RETRIES:
                raise RuntimeError(
                    f"analyze_and_normalize_with_llm fallito dopo {MAX_LLM_RETRIES} tentativi. "
                    f"Ultimo errore: {e}\nRaw: {raw}"
                )
            time.sleep(LLM_RETRY_DELAY)

    normalization = result["normalization"]

    if "original_dialect" not in normalization:
        normalization["original_dialect"] = transcript_whisper

    return normalization


print("✅ analyze_and_normalize_with_llm senza analisi sintattica definita")

✅ analyze_and_normalize_with_llm senza analisi sintattica definita


# Cella 13 - Modulo Claim Spicy

In [20]:
nlp = spacy.load("it_core_news_lg")

# ─────────────────────────────────────────────────────────────
# STEP 1: parsing sintattico → candidate spans
# ─────────────────────────────────────────────────────────────

def extract_spans_from_spacy(normalized_text: str) -> list:
    """
    Analizza la frase normalizzata con spaCy e restituisce
    una lista di span (start_token, end_token, testo del claim).

    Logica: ogni token ROOT individua un nucleo proposizionale.
    Raccogliamo i suoi dipendenti (subtree) come span del claim.
    """
    doc = nlp(normalized_text)

    spans = []
    for token in doc:
        # Ogni ROOT verbale (o ROOT principale) è un nucleo di claim
        if token.dep_ == "ROOT":
            subtree_tokens = sorted(token.subtree, key=lambda t: t.i)

            # Escludiamo punteggiatura e spazi
            filtered = [t for t in subtree_tokens if not t.is_punct and not t.is_space]
            if not filtered:
                continue

            start_i = filtered[0].i
            end_i   = filtered[-1].i
            claim_text = " ".join(t.text for t in filtered)

            spans.append({
                "claim_text":    claim_text,
                "start_token_i": start_i,
                "end_token_i":   end_i,
                "n_tokens":      len(filtered),
            })

    # Se spaCy trova un solo ROOT (frase semplice), splittiamo
    # sulle congiunzioni coordinate ("e", "ma", "però", ",")
    if len(spans) == 1:
        spans = _split_on_conjunctions(doc, spans[0])

    return spans


def _split_on_conjunctions(doc, single_span: dict) -> list:
    """
    Fallback: se c'è un solo span, proviamo a dividerlo
    sui token di coordinazione (cc) o punteggiatura (,).
    """
    split_indices = [
        t.i for t in doc
        if t.dep_ in ("cc", "punct") and t.text in (",", "e", "ma", "però", "quindi", "poi")
    ]

    if not split_indices:
        return [single_span]

    # Costruiamo i sotto-span attorno ai punti di split
    tokens = [t for t in doc if not t.is_punct and not t.is_space]
    result = []
    prev = 0
    for si in split_indices:
        chunk = [t for t in tokens if t.i < si and t.i >= prev]
        if chunk:
            result.append({
                "claim_text":    " ".join(t.text for t in chunk),
                "start_token_i": chunk[0].i,
                "end_token_i":   chunk[-1].i,
                "n_tokens":      len(chunk),
            })
        prev = si + 1

    # Ultimo chunk dopo l'ultimo split
    chunk = [t for t in tokens if t.i >= prev]
    if chunk:
        result.append({
            "claim_text":    " ".join(t.text for t in chunk),
            "start_token_i": chunk[0].i,
            "end_token_i":   chunk[-1].i,
            "n_tokens":      len(chunk),
        })

    return result if result else [single_span]

# Cella 14 - Analisi LLM dei claim

In [21]:
def _compute_asr_signals_from_words(words: list) -> dict:
    """
    Calcola i segnali ASR per-claim dalle parole Whisper allineate.

    Versione SENZA analisi sintattica.

    Segnali:
      conf_min_worst  : confidenza minima del claim
      conf_mean       : media aritmetica delle confidence
      conf_geo_period : media geometrica delle confidence
    """

    if not words:
        return {
            "conf_min_worst":  0.5,
            "conf_mean":       0.5,
            "conf_geo_period": 0.5,
        }

    conf_values = [
        float(w.get("conf_min", w.get("conf_mean", 1.0)))
        for w in words
    ]

    clipped_conf = [max(c, 1e-9) for c in conf_values]
    conf_geo = round(float(np.exp(np.mean(np.log(clipped_conf)))), 4)

    return {
        "conf_min_worst":  round(min(conf_values), 4),
        "conf_mean":       round(float(np.mean(conf_values)), 4),
        "conf_geo_period": conf_geo,
    }


def validate_claims_with_llm(
    claim_list: list,
    normalized_text: str,
    word_data: list,
) -> list:
    """
    Valida semanticamente i claim e riallinea ogni claim alle parole Whisper.

    Versione SENZA:
      - RAG
      - analisi sintattica
      - is_dialectal
      - dialectal_type
      - surprisal

    Aggiunge/aggiorna:
      - claim_type
      - is_question
      - ambiguous
      - ambiguity_reason
      - normalized_claim
      - source_word_indices
      - source_words
      - source_words_data
      - asr_signals
    """

    claims_for_llm = [
        {
            "id": i,
            "claim": c["claim_text"],
        }
        for i, c in enumerate(claim_list)
    ]

    words_for_llm = [
        {
            "idx": i,
            "word": w.get("word", ""),
            "conf_mean": round(float(w.get("conf_mean", 1.0)), 4),
            "conf_min": round(float(w.get("conf_min", 1.0)), 4),
        }
        for i, w in enumerate(word_data)
    ]

    system_prompt = """
You are an expert linguistic and medical semantic analyst specialized in Italian and Southern Italian dialectal ASR transcripts.

Your task is to analyze atomic claims extracted from a normalized Italian sentence and align each claim to the original Whisper word list.

You must perform TWO tasks for each claim:

1. Semantic validation:
   - classify the claim type
   - determine whether it is a question/request
   - detect ambiguity caused by dialect, ASR uncertainty, or unclear medical wording

2. Word alignment:
   - select the exact words from the ordered Whisper word list that semantically support the claim
   - return their integer indices in source_word_indices


CRITICAL ALIGNMENT RULES:
- Use the ordered Whisper word list as the only source for source_word_indices.
- source_word_indices must contain only valid integer idx values from the provided word list.
- Preserve word order.
- Do not assign the same word to multiple claims unless the word is genuinely shared by both claims.
- Prefer semantic alignment over purely positional alignment.
- Include dialectal words when they express the same meaning as the normalized claim.
- Include uncertain or low-confidence words if they are part of the claim.
- Do not include filler words unless they are necessary for the meaning.
- If a claim cannot be aligned confidently, return the best minimal alignment and set ambiguous=true.

MEDICAL SAFETY RULES:
- Do not invent medication names.
- If the medication/symptom/body part is uncertain, set ambiguous=true and explain in ambiguity_reason.
- Do not output entities. The field entities is forbidden.

ALLOWED claim_type values:
- "sintomo"
- "richiesta_farmaco"
- "condizione_medica"
- "azione"
- "domanda_generica"
- "informazione"

OUTPUT RULES:
- Return only a valid JSON array.
- No markdown.
- No backticks.
- No explanatory text outside JSON.

REQUIRED OUTPUT FIELDS FOR EACH CLAIM:
- "id": same id received
- "claim_type": one allowed value
- "is_question": boolean
- "ambiguous": boolean
- "ambiguity_reason": string, empty if ambiguous=false
- "source_word_indices": list of integer indices from the Whisper-RAG word list

Example:
[
  {
    "id": 0,
    "claim_type": "sintomo",
    "is_question": false,
    "ambiguous": false,
    "ambiguity_reason": "",
    "source_word_indices": [0, 1, 2, 3]
  }
]

"""

    user_prompt = f"""
NORMALIZED SENTENCE:
{normalized_text}

CLAIMS TO ANALYZE:
{json.dumps(claims_for_llm, ensure_ascii=False)}

ORDERED WHISPER WORD LIST:
{json.dumps(words_for_llm, ensure_ascii=False)}

Return only the JSON array.
"""

    raw = call_llm_with_retry(
        prompt=user_prompt,
        system_prompt=system_prompt,
        max_tokens=2048,
        temperature=0.0,
    )

    raw = raw.strip()
    if raw.startswith("```json"):
        raw = raw[7:]
    if raw.startswith("```"):
        raw = raw[3:]
    if raw.endswith("```"):
        raw = raw[:-3]
    raw = raw.strip()

    try:
        llm_validation = json.loads(raw)
    except json.JSONDecodeError:
        llm_validation = json_repair.loads(raw)

    if not isinstance(llm_validation, list):
        raise ValueError("La risposta LLM non è una lista JSON")

    llm_by_id = {item.get("id"): item for item in llm_validation}
    max_idx = len(word_data) - 1

    enriched_claims = []

    for i, claim in enumerate(claim_list):
        llm_data = llm_by_id.get(i, {})

        raw_indices = llm_data.get("source_word_indices", [])
        clean_indices = []

        for idx in raw_indices:
            if isinstance(idx, int) and 0 <= idx <= max_idx and idx not in clean_indices:
                clean_indices.append(idx)

        source_words_data = [word_data[idx] for idx in clean_indices]
        source_words = [w.get("word", "") for w in source_words_data]

        asr_signals = _compute_asr_signals_from_words(source_words_data)

        enriched_claims.append({
            **claim,
            "claim_type": llm_data.get("claim_type", "domanda_generica"),
            "is_question": llm_data.get("is_question", True),
            "ambiguous": llm_data.get("ambiguous", False),
            "ambiguity_reason": llm_data.get("ambiguity_reason", ""),
            "source_word_indices": clean_indices,
            "source_words": source_words,
            "source_words_data": source_words_data,
            "asr_signals": asr_signals,
            "pppl_claim": None,
            "pppl_claim_log": None,
            "pppl_claim_tokens": None,
        })

    return enriched_claims


print("✅ validate_claims_with_llm senza analisi sintattica definita")

✅ validate_claims_with_llm senza analisi sintattica definita


# Cella 15 - Calcolo Risk Score

In [24]:
# Pesi aliquota TRASCRIZIONE per claim_type  [geo-first]
# I 3 segnali ASR: tutti già in [0,1] dove 1 = rischio massimo
# (conf_min_worst e conf_mean vengono invertiti: (1 - conf) = rischio)
# conf_geo_period: media geometrica sul periodo → segnale primario

WEIGHTS_TRANSCRIPT_BY_CLAIM_TYPE = {

    # Farmaco: conf_geo domina perché la trascrizione del nome del farmaco
    # impatta più token contigui (es. "amoxicillina" → 3-4 token).
    # conf_min_worst resta a 0.30 come guardia per OOV/token isolati.

    "richiesta_farmaco": {
        "conf_min_worst":  0.30,
        "conf_mean":       0.28,
        "conf_geo_period": 0.42,
    },

    # Sintomo: frasi perifrastiche lunghe in napoletano → geo cattura
    # la qualità complessiva meglio del minimo puntuale.

    "sintomo": {
        "conf_min_worst":  0.25,
        "conf_mean":       0.30,
        "conf_geo_period": 0.45,
    },

    # Condizione medica: simile a sintomo, ma con guardia leggermente
    # più alta su conf_min_worst per termini diagnostici specifici.

    "condizione_medica": {
        "conf_min_worst":  0.28,
        "conf_mean":       0.30,
        "conf_geo_period": 0.42,
    },

    # Azione: claim tendenzialmente brevi (verbo + complemento),
    # geo e mean si equivalgono; min ridotto al minimo.

    "azione": {
        "conf_min_worst":  0.22,
        "conf_mean":       0.33,
        "conf_geo_period": 0.45,
    },

    # Domanda generica: bassa criticità clinica, geo domina nettamente
    # per smoothing su frasi intrrogative spesso lunghe e dialettali.

    "domanda_generica": {
        "conf_min_worst":  0.20,
        "conf_mean":       0.35,
        "conf_geo_period": 0.45,

    },
    # Informazione: analogo a domanda generica.
    "informazione": {
        "conf_min_worst":  0.22,
        "conf_mean":       0.33,
        "conf_geo_period": 0.45,
    },
}

DEFAULT_WEIGHTS_TRANSCRIPT = {
    "conf_min_worst":  0.25,
    "conf_mean":       0.30,
    "conf_geo_period": 0.45,
}


# ─────────────────────────────────────────────────────────────
# Peso α (trascrizione vs traduzione) per claim_type
# α alto → ASR domina; α basso → PPPL_claim domina
# ─────────────────────────────────────────────────────────────
ALPHA_BY_CLAIM_TYPE = {
    "richiesta_farmaco": 0.80,
    "sintomo":           0.70,
    "condizione_medica": 0.75,
    "azione":            0.65,
    "domanda_generica":  0.65,
    "informazione":      0.70,
    "_default":          0.70,
}
 
# Soglie semaforo
THRESHOLDS = {
    "green":  0.35,
    "yellow": 0.60,
}
 


def compute_risk_score(claim: dict) -> dict:
    """
    Calcola il risk score per un singolo claim.

    Aliquota 1 — TRASCRIZIONE:
      R_transcript usa solo segnali Whisper:
        - 1 - conf_min_worst
        - 1 - conf_mean
        - 1 - conf_geo_period

    Aliquota 2 — TRADUZIONE:
      R_translation usa pppl_claim normalizzato.

    Combinazione:
      risk = alpha * R_transcript + (1 - alpha) * R_translation

    Per evitare che una PPPL molto bassa renda il sistema troppo ottimista:
      risk >= 0.85 * R_transcript
    """

    sig = claim["asr_signals"]

    claim_type = claim.get("claim_type", "_default")
    w_transcript = WEIGHTS_TRANSCRIPT_BY_CLAIM_TYPE.get(
        claim_type,
        DEFAULT_WEIGHTS_TRANSCRIPT,
    )

    alpha = ALPHA_BY_CLAIM_TYPE.get(
        claim_type,
        ALPHA_BY_CLAIM_TYPE["_default"],
    )

    conf_min = float(sig.get("conf_min_worst", 0.5))
    conf_mean = float(sig.get("conf_mean", 0.5))
    conf_geo = float(sig.get("conf_geo_period", conf_mean))

    R_transcript = (
        (1 - conf_min)  * w_transcript["conf_min_worst"] +
        (1 - conf_mean) * w_transcript["conf_mean"] +
        (1 - conf_geo)  * w_transcript["conf_geo_period"]
    )

    R_transcript = float(np.clip(R_transcript, 0.0, 1.0))

    pppl_claim_raw = claim.get("pppl_claim")
    pppl_claim_norm = _normalize_pppl(
        pppl_claim_raw,
        n_tokens=claim.get("pppl_claim_tokens"),
    )

    R_translation = pppl_claim_norm

    risk = alpha * R_transcript + (1 - alpha) * R_translation

    risk = max(risk, 0.85 * R_transcript)
    risk = float(np.clip(risk, 0.0, 1.0))

    if risk < THRESHOLDS["green"]:
        risk_level = "green"
    elif risk < THRESHOLDS["yellow"]:
        risk_level = "yellow"
    else:
        risk_level = "red"

    contributions_transcript = {
        "conf_min_worst": round((1 - conf_min) * w_transcript["conf_min_worst"], 4),
        "conf_mean": round((1 - conf_mean) * w_transcript["conf_mean"], 4),
        "conf_geo_period": round((1 - conf_geo) * w_transcript["conf_geo_period"], 4),
    }

    top_transcript = max(
        contributions_transcript,
        key=contributions_transcript.get,
    )

    return {
        "risk_score": round(risk, 4),
        "risk_level": risk_level,
        "alpha": round(alpha, 2),
        "R_transcript": round(R_transcript, 4),
        "R_translation": round(R_translation, 4),
        "contributions_transcript": contributions_transcript,
        "top_contributor_transcript": top_transcript,
        "pppl_claim_raw": pppl_claim_raw,
        "pppl_claim_norm": round(pppl_claim_norm, 4),
    }


def score_all_claims(validated_claims: list) -> list:
    """
    Applica compute_risk_score a ogni claim.
    """

    scored = []

    for claim in validated_claims:
        risk_data = compute_risk_score(claim)
        scored.append({
            **claim,
            **risk_data,
        })

    return scored


def compute_overall_risk(scored_claims: list) -> dict:
    """
    Calcola l'overall risk a partire dai colori dei claim,
    pesati in base al claim_type.

    Non restituisce overall_risk_score.
    Restituisce solo:
      - overall_risk_level
      - overall_reason
    """

    if not scored_claims:
        return {
            "overall_risk_level": "green",
            "overall_reason": "Nessun claim presente: rischio complessivo impostato a green."
        }

    claim_type_weights = {
        "richiesta_farmaco":  1.5,
        "sintomo":            1.5,
        "condizione_medica":  1.5,
        "azione":             1.0,
        "informazione":       0.8,
        "domanda_generica":   0.7,
    }

    critical_types = {
        "richiesta_farmaco",
        "sintomo",
        "condizione_medica",
    }

    total_weight = 0.0
    red_weight = 0.0
    yellow_weight = 0.0

    n_red = 0
    n_yellow = 0
    n_green = 0

    has_critical_red = False
    critical_red_types_found = []

    for claim in scored_claims:
        claim_type = claim.get("claim_type", "informazione")
        risk_level = claim.get("risk_level", "green")

        w = claim_type_weights.get(claim_type, 1.0)
        total_weight += w

        if risk_level == "red":
            red_weight += w
            n_red += 1

            if claim_type in critical_types:
                has_critical_red = True
                critical_red_types_found.append(claim_type)

        elif risk_level == "yellow":
            yellow_weight += w
            n_yellow += 1

        elif risk_level == "green":
            n_green += 1

    if total_weight == 0:
        return {
            "overall_risk_level": "green",
            "overall_reason": "Peso totale nullo: rischio complessivo impostato a green."
        }

    p_red = red_weight / total_weight
    p_yellow = yellow_weight / total_weight
    p_risky = (red_weight + yellow_weight) / total_weight

    n_claims = len(scored_claims)

    if has_critical_red:
        overall_level = "red"
        reason = (
            "Overall impostato a red perché è presente almeno un claim red "
            f"di tipo clinicamente critico: {sorted(set(critical_red_types_found))}."
        )

    elif n_claims <= 3 and n_red >= 1:
        overall_level = "red"
        reason = (
            "Overall impostato a red perché il numero di claim è basso "
            f"({n_claims}) ed è presente almeno un claim red."
        )

    elif p_red >= 0.20:
        overall_level = "red"
        reason = (
            "Overall impostato a red perché la quota pesata di claim red "
            f"è {p_red:.2%}, quindi supera la soglia del 20%."
        )

    elif p_risky >= 0.40:
        overall_level = "yellow"
        reason = (
            "Overall impostato a yellow perché la quota pesata di claim non-green "
            f"(yellow + red) è {p_risky:.2%}, quindi supera la soglia del 40%."
        )

    elif p_yellow >= 0.25:
        overall_level = "yellow"
        reason = (
            "Overall impostato a yellow perché la quota pesata di claim yellow "
            f"è {p_yellow:.2%}, quindi supera la soglia del 25%."
        )

    else:
        overall_level = "green"
        reason = (
            "Overall impostato a green perché non sono presenti condizioni sufficienti "
            "per classificare il rischio complessivo come yellow o red."
        )

    return {
        "overall_risk_level": overall_level,
        "overall_reason": reason,
    }
    
print("✅ Risk scoring senza analisi sintattica definito")

✅ Risk scoring senza analisi sintattica definito


# Cella 16 - Esecuzione Pipeline

In [27]:
# ═══════════════════════════════════════════════════════════════
# PIPELINE COMPLETA — esegue tutta la catena per ogni audio
# e produce per ciascun file due report (.json + .txt) in
# /content/reports/<nome_audio_senza_ext>/
# ═══════════════════════════════════════════════════════════════


SEP = "=" * 60

class _ReportLog:
    def __init__(self):
        self.lines = []
    def __call__(self, *args, sep=" ", end="\n"):
        line = sep.join(str(a) for a in args)
        print(line, end=end)
        self.lines.append(line + ("" if end == "\n" else end))
    def section(self, title):
        bar = "=" * 70
        self(bar)
        self(title)
        self(bar)
    def text(self):
        return "".join(self.lines) if any(l.endswith("\n") for l in self.lines) \
               else "\n".join(self.lines)


# ── Loop principale su tutti i file audio caricati ───────────
all_reports_summary = []

for _audio_idx, (PERCORSO_AUDIO, recording_doc) in enumerate(
        zip(audio_files, audio_docs), start=1):
    log = _ReportLog()
    log.section(f"AUDIO {_audio_idx}/{len(audio_files)}: {os.path.basename(PERCORSO_AUDIO)}")

    report = {
        "audio_file":     os.path.basename(PERCORSO_AUDIO),
        "audio_path":     PERCORSO_AUDIO,
        "timestamp":      _now_ms(),
        "pipeline_steps": {},
        "errors":         [],
    }



    # Variabili locali per raccogliere i risultati di ogni step
    # prima di qualsiasi scrittura su MongoDB
    risultato_whisper       = None
    dati_analisi_full   = None
    output_fonetico     = None
    risultato_ensemble  = None
    pppl_result         = None   # PPPL sul testo italiano normalizzato
    claim_list          = None
    validated_claims    = None
    scored_claims       = None
    overall             = None


    session_id = None  # ← inizializza prima del try

    try:
        session_id = init_new_session(recording_doc=recording_doc)

        # ════════════════════════════════════════════════════════
        # STEP 1 — TRASCRIZIONE WHISPER
        # ════════════════════════════════════════════════════════
        t_s1 = _now_ms()
        audio, sr = librosa.load(PERCORSO_AUDIO, sr=16_000, mono=True)
        risultato_whisper = pipeline_whisper.trascrivi(audio, verbose=False)
        t_e1 = _now_ms()

        log("")
        log("=" * 55)
        log("RIEPILOGO WHISPER STANDARD")
        log("=" * 55)
        log(f"Trascrizione: {risultato_whisper['testo_finale']}")
        log("")

        # ════════════════════════════════════════════════════════
        # STEP 2 — ANALISI CONFIDENZA TOKEN/PAROLA
        # ════════════════════════════════════════════════════════
        result = risultato_whisper["result"]
        testo_per_analisi = risultato_whisper["testo_finale"]

        log("")
        log("=" * 60)
        log("TRASCRIZIONE:")
        log(testo_per_analisi)
        log("=" * 60)
        log(f"Confidence media aritmetica : {result['period_conf_mean']:.4f}")
        log(f"Confidence media geometrica : {result['period_conf_geo']:.4f}")
        log("=" * 60)

        df_tokens = pd.DataFrame(result["tokens"])
        log("")
        log("CONFIDENCE PER TOKEN (prime 30):")
        log(df_tokens[["token_text", "confidence"]].head(30).to_string(index=False))

        THRESHOLD = 0.70
        low_conf  = df_tokens[df_tokens["confidence"] < THRESHOLD]
        log("")
        log(f"Token con confidence < {THRESHOLD}:")
        log(low_conf[["token_text", "confidence"]].to_string(index=False))

        t_s2 = _now_ms()
        word_data  = tokens_to_words(result["tokens"])
        t_e2 = _now_ms()

        df_words   = pd.DataFrame(word_data)

        log("")
        log("CONFIDENCE PER PAROLA:")
        log(df_words[["word", "n_tokens", "conf_mean", "conf_min", "conf_geo"]].to_string(index=False))

        low_conf_words = df_words[df_words["conf_mean"] < THRESHOLD]
        log("")
        log(f"Parole con conf_mean < {THRESHOLD}:")
        log(low_conf_words[["word", "conf_mean", "conf_min", "tokens"]].to_string(index=False))

        report["pipeline_steps"]["confidence_analysis"] = {
            "trascrizione":     testo_per_analisi,
            "period_conf_mean": float(result["period_conf_mean"]),
            "period_conf_geo":  float(result["period_conf_geo"]),
            "tokens":           result["tokens"],
            "words":            word_data,
            "threshold_used":   THRESHOLD,
            "low_conf_tokens":  low_conf.to_dict(orient="records"),
            "low_conf_words":   low_conf_words.to_dict(orient="records"),
        }

        dati_analisi_full = {
            "period_conf_mean": result["period_conf_mean"],
            "period_conf_geo":  result["period_conf_geo"],
            "threshold_used":   THRESHOLD,
            "tokens":           result["tokens"],
            "words":            word_data,
            "low_conf_tokens":  low_conf.to_dict(orient="records"),
            "low_conf_words":   low_conf_words.to_dict(orient="records"),
        }

        # ════════════════════════════════════════════════════════
        # STEP 3 — TRASCRIZIONE FONETICA (PhoneticXeus)
        # ════════════════════════════════════════════════════════
        t_s3 = _now_ms()
        output_fonetico = transcribe_with_phonetic_xeus_fixed(PERCORSO_AUDIO, inference)
        t_e3 = _now_ms()

        log("")
        log("--- RISULTATO FONETICO ---")
        log(f"Trascrizione IPA: {output_fonetico}")
        log("--------------------------")

        report["pipeline_steps"]["phonetic_xeus"] = {
            "trascrizione_ipa": output_fonetico,
        }

        # ════════════════════════════════════════════════════════
        # STEP 4+5 — ANALISI SEMANTICA LLM
        # Versione senza analisi sintattica e senza RAG
        # ════════════════════════════════════════════════════════
        
        t_s4 = _now_ms()
        
        risultato_ensemble = analyze_and_normalize_with_llm(
            transcript_whisper      = testo_per_analisi,
            transcript_PhoneticXeus = output_fonetico,
            word_data               = word_data,
        )
        
        t_e5 = _now_ms()
        
        log("")
        log("ANALISI SEMANTICA LLM:")
        log("  Analisi sintattica parola-per-parola disattivata.")
        log("  Pipeline senza RAG: il LLM usa Whisper, confidence e IPA.")
        
        report["pipeline_steps"]["llm_semantic_analysis"] = {
            "syntactic_analysis_enabled": False,
            "rag_enabled": False,
        }
        
        log("")
        log("=" * 60)
        log("🎯 RISULTATO ENSEMBLE (Whisper + PhoneticXeus + LLM):")
        log("=" * 60)
        log(f"Trascrizione Whisper      : {testo_per_analisi}")
        log(f"Trascrizione PhoneticXeus : {output_fonetico}")
        log("-" * 60)
        log(f"Italiano Standard         : {risultato_ensemble.get('normalized_text', 'Dato mancante')}")
        log(f"Dominio                   : {risultato_ensemble.get('detected_domain', risultato_ensemble.get('domain', 'Non specificato'))}")
        
        log("")
        log("Problemi Risolti/Rilevati:")
        issues = risultato_ensemble.get("semantic_issues", [])
        
        if not issues:
            log("  Nessun problema rilevato o lista mancante.")
        else:
            for issue in issues:
                words = issue.get("words", [issue.get("word", "Sconosciuta")])
                indices = issue.get("word_indices", [issue.get("word_index", "Indice mancante")])
                reason = issue.get("reason", "Nessuna spiegazione fornita")
        
                words_str = ", ".join(words)
                indices_str = ", ".join(map(str, indices))
        
                log(f"  - [{words_str}] indici={indices_str} → {reason}")
        
        report["pipeline_steps"]["ensemble_llm"] = risultato_ensemble
        
        t_s5 = t_s4
        t_e4 = t_e5

        # ════════════════════════════════════════════════════════
        # STEP 5.1 — PSEUDO-PERPLEXITY DI SESSIONE (testo normalizzato)
        # Calcolata sul normalized_text prodotto dall'LLM (italiano standard).
        # Segnale globale di sessione: misura la fluenza complessiva post-LLM.
        # Salvata in pipeline_stages come "5.1_pppl_session".
        # ════════════════════════════════════════════════════════
        testo_normalizzato_it = risultato_ensemble.get("normalized_text", "")

        t_s51 = _now_ms()
        pppl_session_result = compute_pppl(testo_normalizzato_it, pppl_min=1.98, pppl_max=30)
        t_e51 = _now_ms()

        log("")
        log("📊 STEP 5.1 — PPPL SESSIONE (testo italiano normalizzato)")
        log(f"   Testo analizzato : {testo_normalizzato_it}")
        log(f"   PPPL             : {pppl_session_result['pppl']}  (più bassa = più fluente)")
        log(f"   log2(PPPL)       : {pppl_session_result['log_pppl']}")
        log(f"   Token analizzati : {pppl_session_result['n_tokens']}")
        log(f"   Mean log P       : {pppl_session_result['mean_log_prob']}")
        log(f"   PPPL_norm        : {pppl_session_result['pppl_norm']}")
        log(f"   Modello BERT     : {pppl_session_result['model']}")
        log(f"   Tempo (ms)       : {t_e51 - t_s51}")


        # ════════════════════════════════════════════════════════
        # STEP 6 — CLAIM SEGMENTATION
        # ════════════════════════════════════════════════════════

        t_s6 = _now_ms()

        claim_list = extract_spans_from_spacy(risultato_ensemble["normalized_text"])

        t_e6 = _now_ms()

        log("")
        log("=" * 60)
        log("CLAIM LIST ESTRATTA CON SPACY")
        log("=" * 60)

        for i, claim in enumerate(claim_list):
            log(f"\n📌 Claim {i+1}: \"{claim['claim_text']}\"")

        report["pipeline_steps"]["claim_extraction"] = {
            "claim_list": claim_list,
        }

        # ════════════════════════════════════════════════════════
        # STEP 7 — VALIDAZIONE LLM + ALLINEAMENTO PAROLE + SEGNALI ASR
        # ════════════════════════════════════════════════════════

        t_s7 = _now_ms()

        validated_claims = validate_claims_with_llm(
            claim_list=claim_list,
            normalized_text=risultato_ensemble["normalized_text"],
            word_data=word_data,
        )

        t_e7 = _now_ms()

        # ════════════════════════════════════════════════════════
        # STEP 7.1 — PPPL PER-CLAIM (sul normalized_claim)
        # Calcolata DOPO validate_claims_with_llm perché normalized_claim
        # è prodotto dall'LLM di validazione — non esiste prima.
        # È il segnale dell'aliquota TRADUZIONE nel risk score.
        # I risultati vengono iniettati direttamente in ogni claim validato.
        # ════════════════════════════════════════════════════════

        t_s71 = _now_ms()

        log("")
        log("=" * 60)
        log("STEP 7.1 — PPPL PER-CLAIM (aliquota traduzione)")
        log("=" * 60)

        for vc in validated_claims:
            # normalized_claim è il testo normalizzato prodotto dall'LLM (STEP 7)
            # È sempre disponibile qui — fallback a claim_text solo per sicurezza
            text_for_pppl = vc.get("claim_text", "")
            pppl_c = compute_pppl(text_for_pppl, pppl_min=1.66, pppl_max=160)
            vc["pppl_claim"]        = pppl_c.get("pppl")
            vc["pppl_claim_log"]    = pppl_c.get("log_pppl")
            vc["pppl_claim_tokens"] = pppl_c.get("n_tokens")
            log(f"   📌 [{vc['claim_text'][:50]}] → pppl_claim={pppl_c.get('pppl')} "
                f"(n_tok={pppl_c.get('n_tokens')})")

        t_e71 = _now_ms()
        log(f"   Tempo totale PPPL per-claim (ms): {t_e71 - t_s71}")

        log("")
        log("=" * 60)
        log("CLAIM LIST VALIDATA DALL'LLM CON PAROLE ALLINEATE")
        log("=" * 60)

        for i, claim in enumerate(validated_claims):
            sig = claim["asr_signals"]

            log(f"\n📌 Claim {i+1}: \"{claim['claim_text']}\"")
            log(f"   Tipo               : {claim['claim_type']}")
            log(f"   È una domanda?     : {claim['is_question']}")
            log(f"   Ambiguo?           : {claim['ambiguous']}")

            if claim.get("ambiguity_reason"):
                log(f"   Motivo ambiguità   : {claim['ambiguity_reason']}")

            log(f"   Indici parole Whisper  : {claim['source_word_indices']}")
            log(f"   Parole originali   : {claim['source_words']}")
            log(f"   conf_min_worst     : {sig['conf_min_worst']}")
            log(f"   conf_mean          : {sig['conf_mean']}")

        report["pipeline_steps"]["claim_validation"] = {
            "validated_claims": validated_claims,
        }

        # ════════════════════════════════════════════════════════
        # STEP 8 — RISK SCORING
        # ════════════════════════════════════════════════════════

        t_s8 = _now_ms()
        scored_claims = score_all_claims(validated_claims)
        overall       = compute_overall_risk(scored_claims)
        t_e8 = _now_ms()

        EMOJI = {"green": "🟢", "yellow": "🟡", "red": "🔴"}

        log("")
        log("=" * 60)
        log("RISK SCORING PER CLAIM")
        log("=" * 60)
        for i, claim in enumerate(scored_claims):
            emoji = EMOJI[claim["risk_level"]]
            log(f"\n{emoji} Claim {i+1}: \"{claim['claim_text']}\"")
            log(f"   Tipo              : {claim['claim_type']}")
            log(f"   Risk score        : {claim['risk_score']}")
            log(f"   Risk level        : {claim['risk_level'].upper()}")
            log(f"   α (transcript)    : {claim['alpha']}")
            log(f"   R_transcript      : {claim['R_transcript']}")
            log(f"   R_translation     : {claim['R_translation']}")
            log(f"   PPPL claim raw    : {claim['pppl_claim_raw']}")
            log(f"   PPPL claim norm   : {claim['pppl_claim_norm']}")
            log(f"   Top contributor   : {claim['top_contributor_transcript']}")
            log(f"   Contributi ASR    : {claim['contributions_transcript']}")

        log("")
        log("=" * 60)
        emoji_overall = EMOJI[overall["overall_risk_level"]]
        log(f"{emoji_overall} OVERALL RISK: "
            f"{overall['overall_risk_level'].upper()}")
        log(
            f"REASON: {overall['overall_reason']}")
        log("=" * 60)

        report["pipeline_steps"]["risk_scoring"] = {
            "scored_claims": scored_claims,
            "overall":       overall,
        }

        all_reports_summary.append({
            "audio_file":         report["audio_file"],
            "normalized_text":    risultato_ensemble.get("normalized_text"),
            "overall_risk_level": overall["overall_risk_level"],
            "n_claims":           len(scored_claims),
        })


        # ════════════════════════════════════════════════════════
        # STEP 9 — METRICHE VALUTATIVE PIPE
        # ════════════════════════════════════════════════════════

        prompt_text_originale = recording_doc.get("promptText", "")
        testo_tradotto_llm    = risultato_ensemble.get("normalized_text", "")

        translation_metrics = {
            "cosine_similarity": None,
            "bleu":              None,
            "rouge1":            None,
            "rouge2":            None,
            "rougeL":            None,
            "wer":               None,
            "prompt_text":       prompt_text_originale,
            "translated_text":   testo_tradotto_llm,
            "embedding_model":   CONFIG["embedding_model"],
        }

        t_s9 = _now_ms()
        if prompt_text_originale and testo_tradotto_llm:

            # ── 1. COSENO SIMILARITY ─────────────────────────────
            # Semantica profonda — gestisce parafrasi e variazioni dialettali
            # Valore: 0-1, più alto = più simile
            embedder = translation_embedder
            emb_originale = embedder.encode(
                prompt_text_originale,
                normalize_embeddings=True,
                show_progress_bar=False,
            )
            emb_tradotto = embedder.encode(
                testo_tradotto_llm,
                normalize_embeddings=True,
                show_progress_bar=False,
            )
            cosine_score = round(float(np.dot(emb_originale, emb_tradotto)), 4)
            translation_metrics["cosine_similarity"] = cosine_score

            # ── 2. BLEU ──────────────────────────────────────────
            # Misura sovrapposizione di n-grammi tra testo prodotto e riferimento
            # Pensato per machine translation — penalizza variazioni lessicali
            # Valore: 0-100, più alto = più simile
            # NOTA: su testi brevi tende a dare score bassi — usare come segnale
            # relativo tra recording, non come valore assoluto
            bleu_metric = BLEU(effective_order=True)  # effective_order=True gestisce testi brevi
            bleu_result = bleu_metric.sentence_score(
                hypothesis=testo_tradotto_llm,
                references=[prompt_text_originale],
            )
            translation_metrics["bleu"] = round(bleu_result.score, 4)

            # ── 3. ROUGE ─────────────────────────────────────────
            # Misura recall di n-grammi (quanto del testo originale è coperto)
            # ROUGE-1: unigrammi, ROUGE-2: bigrammi, ROUGE-L: sottosequenza comune
            # Valore: 0-1 (precision, recall, F1) — usiamo F1
            # Più robusto di BLEU su testi brevi
            scorer_rouge = rouge_scorer.RougeScorer(
                ["rouge1", "rouge2", "rougeL"],
                use_stemmer=False   # False: italiano non ha stemmer integrato
            )
            rouge_scores = scorer_rouge.score(
                target=prompt_text_originale,
                prediction=testo_tradotto_llm,
            )
            translation_metrics["rouge1"] = round(rouge_scores["rouge1"].fmeasure, 4)
            translation_metrics["rouge2"] = round(rouge_scores["rouge2"].fmeasure, 4)
            translation_metrics["rougeL"] = round(rouge_scores["rougeL"].fmeasure, 4)

            # ── 4. WER (Word Error Rate) ─────────────────────────
            # Misura quante parole differiscono (sostituzioni + inserzioni + cancellazioni)
            # Pensato per ASR — qui lo usiamo per quantificare la distanza lessicale
            # Valore: 0-1+ (può superare 1 se ci sono molte inserzioni)
            # Più basso = più simile; 0 = identici
            wer_score = round(wer(
                reference=prompt_text_originale,
                hypothesis=testo_tradotto_llm,
            ), 4)
            translation_metrics["wer"] = wer_score


            log("")
            log("📐 TRANSLATION QUALITY METRICS")
            log(f"   Cosine Similarity : {cosine_score:.4f}  (0-1,  più alto = meglio)")
            log(f"   BLEU              : {translation_metrics['bleu']:.4f}  (0-100, più alto = meglio)")
            log(f"   ROUGE-1           : {translation_metrics['rouge1']:.4f}  (0-1,  più alto = meglio)")
            log(f"   ROUGE-2           : {translation_metrics['rouge2']:.4f}  (0-1,  più alto = meglio)")
            log(f"   ROUGE-L           : {translation_metrics['rougeL']:.4f}  (0-1,  più alto = meglio)")
            log(f"   WER               : {wer_score:.4f}  (0-1+, più basso = meglio)")

        else:
            log("⚠️  Metriche non calcolate — promptText o traduzione mancanti")

        t_e9 = _now_ms()

        # ════════════════════════════════════════════════════════
        # ✅ SCRITTURA SU MONGODB — solo qui, solo se tutti gli
        #    step precedenti sono andati a buon fine senza errori
        # ════════════════════════════════════════════════════════
        log("")
        log("=" * 60)
        log("SALVATAGGIO SU MONGODB")
        log("=" * 60)

        transcript_results = {
            "mode": "whisper_standard_baseline",
            "testo_finale": risultato_whisper.get("testo_finale"),
            "model": CONFIG.get("whisper_model"),
        }
        
        save_transcript_output(
            session_id,
            transcript_results,
            dati_analisi_full,
            t_start=t_s1,
            t_end=t_e2
        )
        log("  ✅ save_transcript_output")

        save_pipeline_stage(session_id, "3_phonetic_xeus", {"ipa_text": output_fonetico}, t_start=t_s3, t_end=t_e3)
        log("  ✅ save_pipeline_stage: 3_phonetic_xeus")

        save_pipeline_stage(
            session_id,
            "4_llm_semantic_analysis",
            {
                "syntactic_analysis_enabled": False,
                "rag_enabled": False,
            },
            t_start=t_s4,
            t_end=t_e4,
        )

        save_pipeline_stage(session_id, "5_ensemble_llm", risultato_ensemble, t_start=t_s5, t_end=t_e5)
        log("  ✅ save_pipeline_stage: 5_ensemble_llm")

        # ── STEP 5.1: salvataggio PPPL sessione ──────────────────
        save_pipeline_stage(session_id, "5_1_pppl_session", pppl_session_result, t_start=t_s51, t_end=t_e51)
        log("  ✅ save_pipeline_stage: 5_1_pppl_session")

        save_pipeline_stage(session_id, "6_claim_extraction", claim_list, t_start=t_s6, t_end=t_e6)
        log("  ✅ save_pipeline_stage: 6_claim_extraction")

        save_pipeline_stage(session_id, "7_claim_validation", validated_claims, t_start=t_s7, t_end=t_e7)
        log("  ✅ save_pipeline_stage: 7_claim_validation")

        ppl_claims = [{
                "claim_text":       vc.get("claim_text"),
                "pppl_claim":       vc.get("pppl_claim"),
                "pppl_claim_log":   vc.get("pppl_claim_log"),
                "pppl_claim_tokens": vc.get("pppl_claim_tokens"),
            }
            for vc in validated_claims]

        save_pipeline_stage(session_id, "7_1_pppl_claims", ppl_claims, t_start=t_s71, t_end=t_e71)
        log("  ✅ save_pipeline_stage: 7_1_pppl_claims")

        save_risk_scoring(session_id, risk_scores_output=copy.deepcopy(scored_claims), t_start=t_s8, t_end=t_e8)
        log("  ✅ save_risk_scoring")

        save_translation_metrics(session_id, translation_metrics, t_start=t_s9, t_end=t_e9)
        log("  ✅ save_translation_metrics")

        complete_session(session_id, success=True, overall=overall)
        log("  ✅ complete_session → status: completed")

    except Exception as e:
        err = traceback.format_exc()
        log("")
        log("!" * 60)
        log(f"ERRORE durante l'elaborazione di {PERCORSO_AUDIO}:")
        log(err)
        log("!" * 60)
        report["errors"].append({
            "message":   str(e),
            "traceback": err,
        })

        if session_id is not None:          # ← solo se la sessione era stata creata
            complete_session(session_id, success=False, error_msg=str(e))
            log("  ⚠️  MongoDB: sessione marcata come failed")
        else:
            log("  ⚠️  Sessione non inizializzata — nessun dato scritto su MongoDB")

    # ════════════════════════════════════════════════════════════
    # SALVATAGGIO REPORT LOCALE (sempre, anche in caso di errore)
    # ════════════════════════════════════════════════════════════
    stem      = Path(PERCORSO_AUDIO).stem.replace(" ", "_")
    audio_dir = REPORTS_DIR / stem
    audio_dir.mkdir(parents=True, exist_ok=True)

    json_path = audio_dir / f"{stem}__report.json"
    txt_path  = audio_dir / f"{stem}__report.txt"

    with open(json_path, "w", encoding="utf-8") as f:
        _json.dump(report, f, ensure_ascii=False, indent=2, default=_json_default)
    with open(txt_path, "w", encoding="utf-8") as f:
        f.write(log.text())

    print(f"\n💾 Report salvati in:")
    print(f"   {json_path}")
    print(f"   {txt_path}")


# ─── Salvataggio del riepilogo cross-audio ────────────────────
summary_path = REPORTS_DIR / "ALL_AUDIO_SUMMARY.json"
with open(summary_path, "w", encoding="utf-8") as f:
    _json.dump(all_reports_summary, f, ensure_ascii=False, indent=2, default=_json_default)

print(f"\n📊 Riepilogo globale di {len(all_reports_summary)} audio salvato in:")
print(f"   {summary_path}")
print("\n✅ Pipeline completata su tutti i file.")

Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


AUDIO 1/1: prompt-5565_rec-10.wav

RIEPILOGO WHISPER STANDARD
Trascrizione: Negli ultimi 8 anni sono dormito in un ritorno vicino a Gallo, che si è lentamente spalzato. Ho avuto un rinuncio, ho avuto la caviglia della coscia destra, ho scoperto che ho un nervo sciatico, ho avuto un diabete a 10 anni, ho avuto una risonanza magnetica, 8 anni fa ho già mostrato.


TRASCRIZIONE:
Negli ultimi 8 anni sono dormito in un ritorno vicino a Gallo, che si è lentamente spalzato. Ho avuto un rinuncio, ho avuto la caviglia della coscia destra, ho scoperto che ho un nervo sciatico, ho avuto un diabete a 10 anni, ho avuto una risonanza magnetica, 8 anni fa ho già mostrato.
Confidence media aritmetica : 0.6750
Confidence media geometrica : 0.5978

CONFIDENCE PER TOKEN (prime 30):
token_text  confidence
       Neg    0.234159
        li    0.997737
       ult    0.996231
       imi    0.999033
         8    0.537844
      anni    0.992685
      sono    0.536123
      dorm    0.085100
       ito    0.846

In [ ]:
import shutil
import os
from pathlib import Path

# ── CONFIGURAZIONE PERCORSI ────────────────────────────────
# Su Kaggle i report creati nei passi precedenti saranno in /kaggle/working/reports
CARTELLA_DA_SCARICARE = "/kaggle/working/reports"
NOME_ZIP = "download_reports"
# ───────────────────────────────────────────────────────────

# Definiamo il percorso di output nella cartella working di Kaggle
output_zip_path = f"/kaggle/working/{NOME_ZIP}"

# Verifica se la cartella esiste prima di zippare
if os.path.exists(CARTELLA_DA_SCARICARE):
    # Crea l'archivio .zip
    # shutil.make_archive aggiungerà automaticamente l'estensione .zip
    shutil.make_archive(output_zip_path, "zip", CARTELLA_DA_SCARICARE)
    print(f"✅ Archivio creato correttamente: {output_zip_path}.zip")
    print(f"📂 Puoi scaricarlo dal pannello 'Data' -> 'Output' a destra.")
else:
    print(f"❌ Errore: La cartella {CARTELLA_DA_SCARICARE} non esiste.")